Primer Trabajo FSI: Redes Neuronales

Importamos las librerías

In [17]:
import torch
from torch import nn, optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import pandas as pd
import numpy as np
import os
import kagglehub
from torchsummary import summary
import torch.nn.functional as F

Como tenemos una carpeta con las imágenes y otra con las etiquetas (YOLO) lo primero que debemos hacer es modificar este formato para tener un fichero csv con las etiquetas.

In [18]:
# --- Configuración del Dataset de KaggleHub ---
# Reemplaza con el ID de tu dataset en KaggleHub. 
# Formato: 'owner/dataset-slug/version' o 'owner/dataset-slug'
KAGGLE_DATASET_ID = 'pkdarabi/cardetection' 

# 1. Descargar el dataset usando kagglehub
# Esto descargará y descomprimirá el dataset en una ubicación temporal/cache.
print(f"Descargando dataset: {KAGGLE_DATASET_ID}")
# 'download' devuelve la ruta local donde se guardó el dataset.
KAGGLE_DOWNLOAD_PATH = kagglehub.dataset_download(KAGGLE_DATASET_ID)
print(f"Dataset descargado en: {KAGGLE_DOWNLOAD_PATH}")

# --- Configuración de Rutas (Ajustadas a la descarga) ---
# **¡Ajuste crucial!** Reemplaza 'train/images' y 'train/labels' si tu dataset
# tiene una estructura de subcarpeta diferente (ej: 'images', 'labels' directamente).
# La mayoría de los datasets YOLO tienen una carpeta de 'train' o 'data'.

# Definición de las rutas finales
IMAGEN_DIR = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'train', 'images') 
ETIQUETAS_DIR = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'train', 'labels') 
CSV_SALIDA = 'yolo_labels_dataset.csv'
EXTENSION_IMAGEN = '.jpg'
EXTENSION_ETIQUETA = '.txt'

datos = []

print(f"\nEscaneando imágenes en: {IMAGEN_DIR}")

# 2. Iterar sobre los archivos de imagen para emparejar
if not os.path.exists(IMAGEN_DIR):
    print(f"ERROR: No se encontró el directorio de imágenes en {IMAGEN_DIR}. Revisa la estructura del dataset de KaggleHub.")
else:
    for archivo_imagen in os.listdir(IMAGEN_DIR):
        if archivo_imagen.lower().endswith(EXTENSION_IMAGEN):
            # El nombre base (sin extensión) es la clave de emparejamiento
            nombre_base = os.path.splitext(archivo_imagen)[0]
            
            # 3. Construir la ruta al archivo de etiqueta
            ruta_etiqueta = os.path.join(ETIQUETAS_DIR, nombre_base + EXTENSION_ETIQUETA)
            
            if os.path.exists(ruta_etiqueta):
                
                # 4. Leer el archivo de etiqueta
                with open(ruta_etiqueta, 'r') as f:
                    lineas = f.readlines()
                
                # 5. Procesar cada línea (cada objeto/bounding box)
                for linea in lineas:
                    partes = linea.strip().split()
                    
                    if len(partes) == 5:
                        # El primer valor es la CLASE (entero), el resto son coordenadas (flotantes)
                        clase_idx = int(partes[0])
                        x_center = float(partes[1])
                        y_center = float(partes[2])
                        width = float(partes[3])
                        height = float(partes[4])
                        
                        # 6. Almacenar los datos
                        datos.append({
                            'nombre_archivo': archivo_imagen,
                            'clase_indice': clase_idx,
                            'x_center': x_center,
                            'y_center': y_center,
                            'width': width,
                            'height': height
                        })
                    else:
                        print(f"¡Advertencia! Línea con formato incorrecto en {ruta_etiqueta}: {linea.strip()}")
            else:
                # Nota: Es común en detección que algunas imágenes no tengan objetos (no hay archivo .txt)
                # Si esto ocurre, la imagen no tendrá entradas en el CSV (lo cual es correcto).
                pass
                # print(f"¡Advertencia! No se encontró el archivo de etiqueta para: {archivo_imagen}")


# 7. Crear el DataFrame y guardarlo
df = pd.DataFrame(datos)
df.to_csv(CSV_SALIDA, index=False)

print("\n--- Resumen ---")
print(f"Total de cajas delimitadoras encontradas: {len(df)}")
if not df.empty:
    print("Conteo de objetos por clase (Índice):")
    print(df['clase_indice'].value_counts().sort_index())
print(f"¡CSV creado exitosamente en: {CSV_SALIDA}!")

Descargando dataset: pkdarabi/cardetection
Dataset descargado en: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5

Escaneando imágenes en: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\train\images

--- Resumen ---
Total de cajas delimitadoras encontradas: 4298
Conteo de objetos por clase (Índice):
clase_indice
0     542
1     585
2      19
3     267
4     101
5     252
6     285
7     334
8     235
9     283
10    301
11    318
12    323
13    168
14    285
Name: count, dtype: int64
¡CSV creado exitosamente en: yolo_labels_dataset.csv!


Ahora ya podemos crear una instancia de la clase YOLODataset

In [19]:
# --- CÓDIGO PARA GENERAR EL DATASET ---
IMAGE_SIZE = (416, 416) 
DATA_DIR = './'
LABELS_NAME = 'yolo_labels_dataset.csv'

# 1. Definir transformaciones
transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(), 
])


# 2. Clase para Detección de Objetos (SIN CAMBIOS)
class YOLODataset(Dataset):
    def __init__(self, archivo_csv, directorio_imagenes, transform=None):
        self.full_labels_df = pd.read_csv(archivo_csv)
        self.directorio_imagenes = directorio_imagenes
        self.transform = transform

        self.imagenes_unicas = self.full_labels_df['nombre_archivo'].unique()
        self.labels_grouped = self.full_labels_df.groupby('nombre_archivo')

    def __len__(self):
        return len(self.imagenes_unicas)

    def __getitem__(self, idx):
        image_name = self.imagenes_unicas[idx]
        image_path = os.path.join(self.directorio_imagenes, image_name)
        image = Image.open(image_path).convert('RGB')
        
        boxes_df = self.labels_grouped.get_group(image_name)
        
        clase_idx = int(boxes_df['clase_indice'].iloc[0])
        label = torch.tensor(clase_idx, dtype=torch.long)

        if self.transform:
            image = self.transform(image)

        return image, label


# 2. Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv = os.path.join(DATA_DIR, LABELS_NAME) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'train', 'images') 

# 3. Crear instancia del Dataset
if not os.path.exists(ruta_csv) or not os.path.exists(ruta_imgs):
    print("Error: Asegúrate de que el CSV existe y que la ruta de imágenes de Kaggle es correcta.")
else:
    dataset_yolo = YOLODataset(archivo_csv=ruta_csv, directorio_imagenes=ruta_imgs, transform=transform)
    print(f"\nTipo de objeto creado: {type(dataset_yolo)}")
    print(f"Número total de imágenes (longitud del dataset): {len(dataset_yolo)}")



Tipo de objeto creado: <class '__main__.YOLODataset'>
Número total de imágenes (longitud del dataset): 3527


El siguiente paso será crear el DataLoader.

In [20]:
# --- Configuración del Loader ---
BATCH_SIZE = 16 # Tamaño de batch comúnmente usado en detección de objetos.

# Crear el DataLoader
train_loader = DataLoader(dataset_yolo, batch_size=BATCH_SIZE, shuffle=True)

print(f"\nDataLoader creado con éxito. Número de batches: {len(train_loader)}")


DataLoader creado con éxito. Número de batches: 221


A continuación creamos la red neuronal que entrenaremos con el dataset que hemos creado.

In [21]:
class SimpleNN(nn.Module):
    def __init__(self):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(3*416*416, 512) # 
        self.fc2 = nn.Linear(512, 128)    # Capa oculta con 128 neuronas
        self.fc3 = nn.Linear(128, 15)      # Capa de salida con 10 clases (0-14)
        self.activation = nn.Sigmoid()        # Función de activación Sigmoide
        self.softmax = nn.Softmax(dim=1)  # Función softmax para la capa de salida

    def forward(self, x):
        # x = x.view(-1, 416*416)            
        x = x.view(x.size(0), -1)       # Aplanar la imagen de 416x416 a un vector de 173056
        #print(x.shape)                  # Mostrar la forma del tensor después de aplanarlo
        x = self.fc1(x)                
        x = self.activation(x)            # Función de activación Sigmoide en la capa oculta
        #print(x.shape)                   # Mostrar la forma del tensor después de la primera capa
        x = self.fc2(x)                  # Capa de salida
        x = self.activation(x)
        x = self.fc3(x)
        #print(x.shape)                   # Mostrar la forma del tensor después de la segunda capa
        x = self.softmax(x)              # Aplicar softmax para obtener probabilidades
        return x

Definimos la función de perdida y el optimizador.

In [22]:
model = SimpleNN()
summary(model, (3, 416, 416)) # Resumen del modelo
# El modelo y las dimension de entrad de los datos


----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 512]     265,814,528
           Sigmoid-2                  [-1, 512]               0
            Linear-3                  [-1, 128]          65,664
           Sigmoid-4                  [-1, 128]               0
            Linear-5                   [-1, 15]           1,935
           Softmax-6                   [-1, 15]               0
Total params: 265,882,127
Trainable params: 265,882,127
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 1.98
Forward/backward pass size (MB): 0.01
Params size (MB): 1014.26
Estimated Total Size (MB): 1016.25
----------------------------------------------------------------


Ahora ya podemos entrenar la red.

In [23]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleNN().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.5)

EPOCHS = 10
for epoch in range(EPOCHS):
    running_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(inputs)
        labels_one_hot = F.one_hot(labels, num_classes=15).float()  # 15 clases
        loss = criterion(outputs, labels_one_hot)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    print(f"[Epoch {epoch + 1}] loss: {running_loss / len(train_loader):.3f}")


[Epoch 1] loss: 0.062
[Epoch 2] loss: 0.061
[Epoch 3] loss: 0.061
[Epoch 4] loss: 0.060
[Epoch 5] loss: 0.060
[Epoch 6] loss: 0.058
[Epoch 7] loss: 0.057
[Epoch 8] loss: 0.056
[Epoch 9] loss: 0.055
[Epoch 10] loss: 0.054


Y por último evaluamos la red.

In [33]:
# --- Configuración de rutas para test ---
IMAGEN_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 
ETIQUETAS_DIR_TEST = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'labels')
CSV_SALIDA_TEST = 'yolo_labels_test.csv'

datos = []

print(f"\nEscaneando imágenes en: {IMAGEN_DIR_TEST}")

# 2. Iterar sobre los archivos de imagen para emparejar
if not os.path.exists(IMAGEN_DIR_TEST):
    print(f"ERROR: No se encontró el directorio de imágenes en {IMAGEN_DIR_TEST}. Revisa la estructura del dataset.")
else:
    for archivo_imagen in os.listdir(IMAGEN_DIR_TEST):
        if archivo_imagen.lower().endswith(EXTENSION_IMAGEN):
            nombre_base = os.path.splitext(archivo_imagen)[0]
            ruta_etiqueta = os.path.join(ETIQUETAS_DIR_TEST, nombre_base + EXTENSION_ETIQUETA)
            
            if os.path.exists(ruta_etiqueta):
                with open(ruta_etiqueta, 'r') as f:
                    lineas = f.readlines()
                
                for linea in lineas:
                    partes = linea.strip().split()
                    if len(partes) == 5:
                        clase_idx = int(partes[0])
                        x_center = float(partes[1])
                        y_center = float(partes[2])
                        width = float(partes[3])
                        height = float(partes[4])
                        datos.append({
                            'nombre_archivo': archivo_imagen,
                            'clase_indice': clase_idx,
                            'x_center': x_center,
                            'y_center': y_center,
                            'width': width,
                            'height': height
                        })
                    else:
                        print(f"¡Advertencia! Línea con formato incorrecto en {ruta_etiqueta}: {linea.strip()}")
            # No se requiere else: es normal que algunas imágenes no tengan etiquetas

# 3. Crear DataFrame y guardar CSV
df_test = pd.DataFrame(datos)
df_test.to_csv(CSV_SALIDA_TEST, index=False)

print("\n--- Resumen Test ---")
print(f"Total de cajas delimitadoras encontradas: {len(df_test)}")
if not df_test.empty:
    print("Conteo de objetos por clase (Índice):")
    print(df_test['clase_indice'].value_counts().sort_index())
print(f"¡CSV de test creado exitosamente en: {CSV_SALIDA_TEST}!")

# 4. Definir las rutas usando la ruta de descarga de Kaggle
ruta_csv_test = os.path.join(DATA_DIR, CSV_SALIDA_TEST) # El CSV se creó en el directorio actual
# IMPORTANTE: Definir la ruta de imágenes APUNTANDO al subdirectorio 'train/images'
ruta_imgs_test = os.path.join(KAGGLE_DOWNLOAD_PATH, 'car', 'test', 'images') 

# Crear DataLoader de test
dataset_yolo_test = YOLODataset(archivo_csv=ruta_csv_test, directorio_imagenes=ruta_imgs_test, transform=transform)
test_loader = DataLoader(dataset_yolo_test, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTest DataLoader creado con {len(test_loader)} batches")



Escaneando imágenes en: C:\Users\Daniel\.cache\kagglehub\datasets\pkdarabi\cardetection\versions\5\car\test\images

--- Resumen Test ---
Total de cajas delimitadoras encontradas: 770
Conteo de objetos por clase (Índice):
clase_indice
0     110
1      94
2       3
3      46
4      21
5      44
6      46
7      60
8      53
9      50
10     45
11     53
12     61
13     34
14     50
Name: count, dtype: int64
¡CSV de test creado exitosamente en: yolo_labels_test.csv!

Test DataLoader creado con 40 batches


In [34]:
# Evaluación de la red
def evaluate(model, test_loader):
    model.eval()  # Poner el modelo en modo evaluación
    correct = 0
    total = test_loader.dataset.__len__()  # Total de muestras en el conjunto de test
    print(f'Total de muestras en el conjunto de test: {total}')
    with torch.no_grad():  # No calcular gradientes
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)  # Mover datos al dispositivo
            outputs = model(inputs)  # Forward pass
            _, predicted = torch.max(outputs.data, 1)  # Obtener las predicciones
            correct += (predicted == labels).sum().item()  # Actualizar el contador de aciertos
    accuracy = 100 * correct / total if total > 0 else 0
    print(f'Accuracy: {accuracy:.2f}%')

In [35]:
evaluate(model, test_loader)

Total de muestras en el conjunto de test: 637
Accuracy: 29.67%
